<a href="https://colab.research.google.com/github/meriemjelassi/ml-streamlit-app/blob/main/webscrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [76]:
# Test rapide dans Python
import requests

url = "https://pharma-shop.tn/957-complements-alimentaires"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

response = requests.get(url, headers=headers)
print(f"Status Code: {response.status_code}")
print(f"Taille réponse: {len(response.text)} caractères")
print(f"Contient 'produit'? {'produit' in response.text.lower()}")
print(f"Contient 'product'? {'product' in response.text.lower()}")

Status Code: 200
Taille réponse: 481457 caractères
Contient 'produit'? True
Contient 'product'? True


In [79]:
import requests
from bs4 import BeautifulSoup
import time
import random
import json
import re

# --- CONFIGURATION ---
categories = {
    "ANIMALERIE": ["https://pharma-shop.tn/1273-votre-animalerie-en-ligne"],
    "ACCESSOIRES": ["https://pharma-shop.tn/1184-accesssoires"],
    "COMPLEMENTS_ALIMENTAIRES": ["https://pharma-shop.tn/957-complements-alimentaires"],
    "COMPRIME": ["https://pharma-shop.tn/957-complements-alimentaires"],
    "NATURE": ["https://pharma-shop.tn/1023-bio-naturel"],
    "OPH-ORL": ["https://pharma-shop.tn/1006-nez-et-oreilles"],
    "POMMADE": ["https://pharma-shop.tn/903-articulations"],
    "TOILETTE": ["https://pharma-shop.tn/987-hygiene"],
    "XEN": ["https://pharma-shop.tn/94_xen"],
    "URIAGE": ["https://pharma-shop.tn/70_uriage"],
    "CUMLAUDE": ["https://pharma-shop.tn/372_cumlaude-lab"],
}

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'fr-FR,fr;q=0.9',
}

def extract_product_data(product_block):
    """Extrait les données d'un bloc produit"""
    # NOM
    name = None
    name_selectors = [
        {'tag': 'h1', 'attrs': {'itemprop': 'name'}},
        {'tag': 'h2', 'attrs': {'itemprop': 'name'}},
        {'tag': 'h3', 'attrs': {'itemprop': 'name'}},
        {'tag': 'h1', 'attrs': {'class': 'h1'}},
        {'tag': ['h1', 'h2', 'h3'], 'attrs': {'class': 'product-title'}},
        {'tag': 'a', 'attrs': {'class': 'product-name'}},
    ]

    for selector in name_selectors:
        tag = selector['tag']
        if isinstance(tag, list):
            for t in tag:
                elem = product_block.find(t, selector['attrs'])
                if elem:
                    name = elem.get_text(strip=True)
                    break
            if name:
                break
        else:
            elem = product_block.find(tag, selector['attrs'])
            if elem:
                name = elem.get_text(strip=True)
                break

    # PRIX
    price = None
    price_selectors = [
        {'tag': 'span', 'attrs': {'itemprop': 'price'}},
        {'tag': 'span', 'attrs': {'class': 'price'}},
        {'tag': 'div', 'attrs': {'class': 'product-price'}},
        {'tag': 'span', 'attrs': {'class': re.compile(r'.*price.*', re.I)}},
    ]

    for selector in price_selectors:
        elem = product_block.find(selector['tag'], selector['attrs'])
        if elem:
            price_text = elem.get_text(strip=True)
            price_text = re.sub(r'\s+TTC$', '', price_text, flags=re.I)
            price_text = price_text.replace('&nbsp;', ' ').replace('\xa0', ' ')
            price = price_text.strip()
            break

    return name, price

def scrape_category_page(url, category_name):
    """Scrape une page de catégorie"""
    print(f"   📄 Page: {url}")

    try:
        time.sleep(random.uniform(1, 2))
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        # STRATÉGIES POUR TROUVER LES PRODUITS
        products = []

        # 1. Microdata
        products = soup.find_all('article', itemprop="itemListElement")

        # 2. Classe product-miniature
        if not products:
            products = soup.find_all('div', class_='product-miniature')

        # 3. Chercher avec "product" dans la classe
        if not products:
            for tag in ['article', 'div', 'li']:
                found = soup.find_all(tag, class_=lambda x: x and 'product' in str(x).lower())
                if found:
                    products = found
                    break

        # Vérifier si c'est une page de pagination normale
        if len(products) == 1:
            # Peut-être une page "pas de produits"
            product_text = str(products[0]).lower()
            if 'aucun' in product_text or 'no product' in product_text:
                return []

        # Extraire les données
        all_products = []
        for product_block in products:
            name, price = extract_product_data(product_block)

            if name and name != "NOM_NON_TROUVE":
                # Extraire aussi le lien du produit
                link_tag = product_block.find('a', href=True)
                product_url = link_tag['href'] if link_tag else url

                # Nettoyer l'URL
                if product_url.startswith('//'):
                    product_url = 'https:' + product_url
                elif product_url.startswith('/'):
                    product_url = 'https://pharma-shop.tn' + product_url

                product_data = {
                    'category': category_name,
                    'name': name,
                    'price': price if price else "Non disponible",
                    'product_url': product_url,
                    'page_url': url
                }
                all_products.append(product_data)

        return all_products

    except Exception as e:
        print(f"   ❌ Erreur: {str(e)[:100]}")
        return []

def scrape_category_with_pagination(base_url, category_name, max_pages=50):
    """Scrape TOUTES les pages d'une catégorie"""
    all_products = []
    page = 1
    empty_page_count = 0  # Compteur de pages vides consécutives

    print(f"\n🔗 Début du scraping: {base_url}")
    print(f"📊 Cible: jusqu'à {max_pages} pages")

    while page <= max_pages:
        # Construire l'URL
        if page == 1:
            current_url = base_url
        else:
            current_url = f"{base_url}?page={page}"

        print(f"   📄 Page {page}: {current_url}")

        products = scrape_category_page(current_url, category_name)

        if not products:
            print(f"   ⚠️  Page {page} vide (0 produits)")
            empty_page_count += 1

            # Si 2 pages vides consécutives, on arrête
            if empty_page_count >= 2:
                print(f"   🛑 2 pages vides consécutives. Arrêt.")
                break
        else:
            empty_page_count = 0  # Réinitialiser le compteur
            print(f"   ✅ {len(products)} produits extraits")

            # Afficher un exemple
            if products:
                sample = products[0]
                name_preview = sample['name'][:40] + "..." if len(sample['name']) > 40 else sample['name']
                print(f"   👀 Exemple: {name_preview}")

            all_products.extend(products)

        page += 1

        # Pause plus longue entre les pages pour éviter le blocage
        if page % 10 == 0:  # Toutes les 10 pages
            wait_time = random.uniform(5, 8)
            print(f"   ⏳ Pause de {wait_time:.1f}s...")
            time.sleep(wait_time)
        else:
            time.sleep(random.uniform(1.5, 3))

    return all_products

def main_complete():
    """Scraping COMPLET de toutes les catégories"""
    print("🚀 SCRAPING COMPLET - PHARMA-SHOP.TN")
    print("="*60)
    print(f"📂 Nombre de catégories: {len(categories)}")
    print("="*60)

    all_data = []
    category_stats = {}

    for category_name, urls in categories.items():
        print(f"\n{'='*50}")
        print(f"📂 CATÉGORIE: {category_name}")
        print(f"{'='*50}")

        category_total = 0

        for url in urls:
            print(f"\n🔗 URL: {url}")
            products = scrape_category_with_pagination(url, category_name, max_pages=100)

            category_total += len(products)
            all_data.extend(products)

            print(f"📊 {len(products)} produits pour cette URL")

        category_stats[category_name] = category_total
        print(f"\n🎯 TOTAL {category_name}: {category_total} produits")

    # RÉSULTATS FINAUX
    print(f"\n{'='*60}")
    print("📊 RÉSULTATS FINAUX - SCRAPING COMPLET")
    print(f"{'='*60}")

    total_all = len(all_data)
    print(f"✅ TOTAL GÉNÉRAL: {total_all} produits")

    if total_all > 0:
        # Sauvegarde JSON
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        json_filename = f'pharma_shop_complet_{timestamp}.json'

        with open(json_filename, 'w', encoding='utf-8') as f:
            json.dump(all_data, f, ensure_ascii=False, indent=2)
        print(f"💾 JSON sauvegardé: {json_filename}")

        # Sauvegarde CSV si pandas est disponible
        try:
            import pandas as pd

            df = pd.DataFrame(all_data)
            csv_filename = f'pharma_shop_complet_{timestamp}.csv'

            # Réorganiser les colonnes
            if 'product_url' in df.columns:
                cols = ['category', 'name', 'price', 'product_url', 'page_url']
                df = df[cols]

            df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
            print(f"📄 CSV sauvegardé: {csv_filename}")

        except ImportError:
            print("ℹ️  Pour exporter en CSV: pip install pandas")

        # STATISTIQUES DÉTAILLÉES
        print(f"\n📈 STATISTIQUES PAR CATÉGORIE:")
        print("-"*40)

        sorted_stats = sorted(category_stats.items(), key=lambda x: x[1], reverse=True)

        for cat, count in sorted_stats:
            percentage = (count / total_all * 100) if total_all > 0 else 0
            print(f"   {cat:25} {count:5} produits ({percentage:.1f}%)")

        # APERÇU DES DONNÉES
        print(f"\n📋 APERÇU DES DONNÉES (10 produits aléatoires):")
        print("-"*80)

        import random as rand
        if len(all_data) > 10:
            sample = rand.sample(all_data, 10)
        else:
            sample = all_data

        for i, item in enumerate(sample, 1):
            cat_short = item['category'][:12]
            name_short = item['name'][:40] + "..." if len(item['name']) > 40 else item['name']
            price_display = item['price'] if item['price'] and item['price'] != "Non disponible" else "N/A"
            print(f"{i:2}. [{cat_short:12}] {name_short:43} - {price_display}")

    else:
        print("❌ Aucune donnée récupérée")

    return all_data, category_stats

def estimate_total_products():
    """Estime le nombre total de produits par catégorie"""
    print("🔍 ESTIMATION DU NOMBRE TOTAL DE PRODUITS")
    print("="*60)

    test_urls = {
        "COMPLEMENTS_ALIMENTAIRES": "https://pharma-shop.tn/957-complements-alimentaires",
        "ANIMALERIE": "https://pharma-shop.tn/1273-votre-animalerie-en-ligne",
        "NATURE": "https://pharma-shop.tn/1023-bio-naturel"
    }

    for cat_name, url in test_urls.items():
        print(f"\n📊 {cat_name}:")

        # Test page 1
        response = requests.get(url, headers=HEADERS)
        soup = BeautifulSoup(response.text, 'html.parser')

        # Chercher le nombre total affiché sur le site
        total_products = None

        # Essayer de trouver un texte comme "912 produits"
        for text in soup.find_all(text=True):
            if 'produit' in text.lower() and any(char.isdigit() for char in text):
                # Extraire les chiffres
                numbers = re.findall(r'\d+', text)
                if numbers:
                    total_products = int(numbers[0])
                    print(f"   📈 Selon le site: {total_products} produits")
                    break

        if not total_products:
            print(f"   ℹ️  Nombre total non trouvé dans la page")

        # Compter les produits sur la première page
        products_page1 = soup.find_all('article', itemprop="itemListElement")
        if not products_page1:
            products_page1 = soup.find_all('div', class_='product-miniature')

        print(f"   📄 Produits sur page 1: {len(products_page1)}")

        # Estimer le nombre de pages
        if total_products and products_page1:
            estimated_pages = total_products // len(products_page1)
            if total_products % len(products_page1) > 0:
                estimated_pages += 1

            print(f"   🧮 Pages estimées: {estimated_pages}")
            print(f"   📦 Produits estimés: {total_products}")

if __name__ == "__main__":
    print("🌐 PHARMA-SHOP.TN - SCRAPER COMPLET")
    print("="*60)

    # Estimation d'abord
    estimate_total_products()

    # Demander confirmation
    print(f"\n{'='*60}")
    print("⚠️  ATTENTION: Le scraping complet peut prendre 30-60 minutes")
    print(f"   et récupérer plusieurs milliers de produits.")

    confirm = input("\n🎯 Voulez-vous lancer le scraping COMPLET? (o/n): ").strip().lower()

    if confirm == 'o' or confirm == 'oui':
        print("\n" + "="*60)
        print("🚀 LANCEMENT DU SCRAPING COMPLET...")
        print("="*60)

        results, stats = main_complete()

        print(f"\n{'='*60}")
        print("✅ SCRAPING TERMINÉ !")
        print(f"📁 Les données sont sauvegardées dans les fichiers .json et .csv")
        print(f"📊 Total: {len(results)} produits récupérés")

    else:
        print("\n❌ Scraping annulé.")
        print("💡 Vous pouvez modifier 'max_pages' dans la fonction pour limiter.")

🌐 PHARMA-SHOP.TN - SCRAPER COMPLET
🔍 ESTIMATION DU NOMBRE TOTAL DE PRODUITS

📊 COMPLEMENTS_ALIMENTAIRES:


/tmp/ipython-input-2879240624.py:314: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  for text in soup.find_all(text=True):


   📈 Selon le site: 2 produits
   📄 Produits sur page 1: 0

📊 ANIMALERIE:
   📈 Selon le site: 2 produits
   📄 Produits sur page 1: 0

📊 NATURE:
   📈 Selon le site: 2 produits
   📄 Produits sur page 1: 0

⚠️  ATTENTION: Le scraping complet peut prendre 30-60 minutes
   et récupérer plusieurs milliers de produits.

🎯 Voulez-vous lancer le scraping COMPLET? (o/n): o

🚀 LANCEMENT DU SCRAPING COMPLET...
🚀 SCRAPING COMPLET - PHARMA-SHOP.TN
📂 Nombre de catégories: 11

📂 CATÉGORIE: ANIMALERIE

🔗 URL: https://pharma-shop.tn/1273-votre-animalerie-en-ligne

🔗 Début du scraping: https://pharma-shop.tn/1273-votre-animalerie-en-ligne
📊 Cible: jusqu'à 100 pages
   📄 Page 1: https://pharma-shop.tn/1273-votre-animalerie-en-ligne
   📄 Page: https://pharma-shop.tn/1273-votre-animalerie-en-ligne
   ✅ 24 produits extraits
   👀 Exemple: ZANILOVE PARFUM FRAMBOISE-PÈCHE 100 ML
   📄 Page 2: https://pharma-shop.tn/1273-votre-animalerie-en-ligne?page=2
   📄 Page: https://pharma-shop.tn/1273-votre-animalerie-en-l

In [81]:
import pandas as pd
import json
import os
from datetime import datetime

def create_raw_csv():
    """
    Crée un CSV brut avec TOUTES les données sans aucun nettoyage.
    Parfait pour l'ETL ensuite.
    """
    print("📊 CRÉATION DU CSV BRUT POUR ETL")
    print("="*60)

    # Chercher tous les fichiers JSON
    json_files = []
    for file in os.listdir('.'):
        if file.startswith('pharma_shop_') and file.endswith('.json'):
            json_files.append(file)

    if not json_files:
        print("❌ Aucun fichier JSON trouvé.")
        return None, None

    # Trier par date de création (du plus récent au plus ancien)
    json_files.sort(key=lambda x: os.path.getctime(x), reverse=True)

    print(f"📁 Fichiers JSON trouvés: {len(json_files)}")
    for i, file in enumerate(json_files[:5], 1):  # Afficher les 5 premiers
        file_time = datetime.fromtimestamp(os.path.getctime(file))
        size_kb = os.path.getsize(file) / 1024
        print(f"   {i}. {file} ({size_kb:.1f} Ko, {file_time.strftime('%Y-%m-%d %H:%M')})")

    if len(json_files) > 5:
        print(f"   ... et {len(json_files) - 5} autres fichiers")

    # Charger TOUTES les données brutes
    all_raw_data = []
    file_stats = {}

    print("\n📥 CHARGEMENT DES DONNÉES BRUTES...")
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                all_raw_data.extend(data)
                file_stats[json_file] = len(data)
                print(f"   ✅ {json_file}: {len(data)} lignes brutes")
        except Exception as e:
            print(f"   ❌ Erreur avec {json_file}: {e}")

    if not all_raw_data:
        print("❌ Aucune donnée à exporter.")
        return None, None

    print(f"\n📊 TOTAL LIGNES BRUTES: {len(all_raw_data)}")

    # Créer le DataFrame SANS MODIFICATION
    df_raw = pd.DataFrame(all_raw_data)

    # Afficher les métadonnées
    print(f"\n🔍 MÉTADONNÉES DES DONNÉES BRUTES:")
    print(f"   • Nombre de colonnes: {len(df_raw.columns)}")
    print(f"   • Colonnes disponibles: {list(df_raw.columns)}")

    # Statistiques par colonne
    print(f"\n📈 STATISTIQUES PAR COLONNE:")
    for col in df_raw.columns:
        non_null = df_raw[col].notna().sum()
        null_count = df_raw[col].isna().sum()
        unique_count = df_raw[col].nunique()

        print(f"   • {col:20} : {non_null:5} non-null, {null_count:5} null, {unique_count:5} valeurs uniques")

        # Afficher un exemple de valeur
        if non_null > 0:
            sample = df_raw[col].dropna().iloc[0]
            sample_str = str(sample)[:50] + "..." if len(str(sample)) > 50 else str(sample)
            print(f"        Ex: {sample_str}")

    # Conserver l'ordre original des colonnes
    original_columns = list(df_raw.columns)

    # Ajouter une colonne avec la source (fichier d'origine)
    df_raw['_source_file'] = ""
    current_index = 0

    for json_file, row_count in file_stats.items():
        df_raw.loc[current_index:current_index + row_count - 1, '_source_file'] = json_file
        current_index += row_count

    # Ajouter un timestamp d'export
    export_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    df_raw['_export_timestamp'] = export_time

    # Réorganiser les colonnes (metadonnées à la fin)
    final_columns = original_columns + ['_source_file', '_export_timestamp']
    df_raw = df_raw[final_columns]

    # Créer le nom du fichier CSV
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_filename = f'pharma_shop_raw_data_{timestamp}.csv'

    # Sauvegarder en CSV SANS MODIFICATION
    df_raw.to_csv(csv_filename, index=False, encoding='utf-8-sig')

    print(f"\n💾 CSV BRUT SAUVEGARDÉ: {csv_filename}")
    print(f"📏 Dimensions: {len(df_raw)} lignes × {len(df_raw.columns)} colonnes")

    # Calculer la taille
    file_size_kb = os.path.getsize(csv_filename) / 1024
    file_size_mb = file_size_kb / 1024
    print(f"📦 Taille du fichier: {file_size_kb:.1f} Ko ({file_size_mb:.2f} Mo)")

    # Aperçu des données brutes (premières 5 lignes)
    print(f"\n📋 APERÇU DES DONNÉES BRUTES (5 premières lignes):")
    print("="*100)

    # Afficher un aperçu complet (sans troncature)
    with pd.option_context('display.max_columns', None,
                          'display.max_rows', 5,
                          'display.width', None,
                          'display.max_colwidth', 50):
        print(df_raw.head())

    # Statistiques par catégorie (si la colonne existe)
    if 'category' in df_raw.columns:
        print(f"\n📊 DISTRIBUTION PAR CATÉGORIE (BRUTE):")
        print("-"*50)

        category_counts = df_raw['category'].value_counts()
        total_categories = len(category_counts)

        print(f"   Nombre de catégories uniques: {total_categories}")

        for category, count in category_counts.head(10).items():
            percentage = (count / len(df_raw)) * 100
            print(f"   • {category:25} : {count:5} lignes ({percentage:5.1f}%)")

        if total_categories > 10:
            other_categories = category_counts.iloc[10:].sum()
            print(f"   • AUTRES ({total_categories - 10} catégories) : {other_categories:5} lignes")

    # Informations pour l'ETL
    print(f"\n🎯 INFORMATIONS POUR VOTRE ETL:")
    print(f"   • Fichier source: {csv_filename}")
    print(f"   • Encodage: UTF-8 avec BOM (utf-8-sig)")
    print(f"   • Séparateur: virgule (CSV standard)")
    print(f"   • En-têtes: Oui")
    print(f"   • Données brutes: Oui (aucun nettoyage appliqué)")
    print(f"   • Métadonnées ajoutées: _source_file, _export_timestamp")

    return df_raw, csv_filename

def export_minimal_raw_csv():
    """
    Version minimale pour un export rapide en CSV brut
    """
    print("📄 EXPORT MINIMAL BRUT")
    print("="*60)

    # Chercher le dernier fichier JSON
    json_files = [f for f in os.listdir('.') if f.startswith('pharma_shop_') and f.endswith('.json')]

    if not json_files:
        print("❌ Aucun fichier JSON trouvé.")
        return None, None

    # Prendre le plus récent
    latest_json = max(json_files, key=os.path.getctime)
    print(f"📂 Chargement du fichier: {latest_json}")

    # Charger les données brutes
    with open(latest_json, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)

    print(f"📊 Données brutes chargées: {len(raw_data)} lignes")

    # Convertir en DataFrame SANS modification
    df_raw = pd.DataFrame(raw_data)

    # Sauvegarder en CSV brut
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_file = f'pharma_shop_raw_export_{timestamp}.csv'

    df_raw.to_csv(csv_file, index=False, encoding='utf-8-sig')

    print(f"\n✅ CSV BRUT CRÉÉ: {csv_file}")
    print(f"📏 Dimensions: {len(df_raw)} lignes × {len(df_raw.columns)} colonnes")

    # Afficher un échantillon
    print("\n📋 ÉCHANTILLON (3 premières lignes):")
    print(df_raw.head(3).to_string(index=False))

    return df_raw, csv_file

# Menu principal
if __name__ == "__main__":
    print("🌐 PHARMA-SHOP.TN - EXPORT BRUT POUR ETL")
    print("="*70)
    print("⚠️  ATTENTION: Aucun nettoyage ne sera appliqué aux données")
    print("   Les données seront exportées telles quelles pour l'ETL.")
    print("="*70)

    print("\n🎯 CHOISISSEZ UNE OPTION:")
    print("   1. CSV brut complet (tous les fichiers JSON)")
    print("   2. CSV brut simple (dernier fichier JSON seulement)")
    print("   3. Quitter")

    choice = input("\nVotre choix (1, 2 ou 3): ").strip()

    if choice == "1":
        print("\n" + "="*70)
        df_raw, filename = create_raw_csv()

        if df_raw is not None:
            print(f"\n✅ EXPORT BRUT TERMINÉ!")
            print(f"📁 Fichier: {filename}")
            print(f"📊 Données brutes: {len(df_raw)} lignes")

            # Demander si on veut voir le chemin complet
            show_path = input("\n📂 Afficher le chemin complet? (o/n): ").strip().lower()
            if show_path == 'o':
                import pathlib
                full_path = pathlib.Path(filename).absolute()
                print(f"📁 Chemin: {full_path}")

    elif choice == "2":
        print("\n" + "="*70)
        df_raw, filename = export_minimal_raw_csv()

        if df_raw is not None:
            print(f"\n✅ EXPORT SIMPLE TERMINÉ!")
            print(f"📁 Fichier: {filename}")

    else:
        print("\n👋 Au revoir!")

🌐 PHARMA-SHOP.TN - EXPORT BRUT POUR ETL
⚠️  ATTENTION: Aucun nettoyage ne sera appliqué aux données
   Les données seront exportées telles quelles pour l'ETL.

🎯 CHOISISSEZ UNE OPTION:
   1. CSV brut complet (tous les fichiers JSON)
   2. CSV brut simple (dernier fichier JSON seulement)
   3. Quitter

Votre choix (1, 2 ou 3): 1

📊 CRÉATION DU CSV BRUT POUR ETL
📁 Fichiers JSON trouvés: 3
   1. pharma_shop_complet_20260201_212207.json (1249.6 Ko, 2026-02-01 21:22)
   2. pharma_shop_complet_20260201_203714.json (1249.6 Ko, 2026-02-01 20:37)
   3. pharma_shop_test.json (35.1 Ko, 2026-02-01 20:09)

📥 CHARGEMENT DES DONNÉES BRUTES...
   ✅ pharma_shop_complet_20260201_212207.json: 4006 lignes brutes
   ✅ pharma_shop_complet_20260201_203714.json: 4006 lignes brutes
   ✅ pharma_shop_test.json: 187 lignes brutes

📊 TOTAL LIGNES BRUTES: 8199

🔍 MÉTADONNÉES DES DONNÉES BRUTES:
   • Nombre de colonnes: 6
   • Colonnes disponibles: ['category', 'name', 'price', 'product_url', 'page_url', 'url']

📈 S

In [82]:
import pandas as pd
import json
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

def export_to_excel_raw():
    """
    Exporte TOUTES les données brutes en fichier Excel (.xlsx)
    Aucun nettoyage - parfait pour ETL
    """
    print("📊 EXPORT BRUT VERS EXCEL")
    print("="*60)

    # Chercher tous les fichiers JSON
    json_files = []
    for file in os.listdir('.'):
        if file.startswith('pharma_shop_') and file.endswith('.json'):
            json_files.append(file)

    if not json_files:
        print("❌ Aucun fichier JSON trouvé.")
        return None, None

    # Trier par date
    json_files.sort(key=lambda x: os.path.getctime(x), reverse=True)

    print(f"📁 Fichiers JSON trouvés: {len(json_files)}")
    for i, file in enumerate(json_files[:3], 1):
        size_mb = os.path.getsize(file) / (1024*1024)
        print(f"   {i}. {file} ({size_mb:.2f} MB)")

    # Charger TOUTES les données brutes
    all_raw_data = []
    print("\n📥 CHARGEMENT DES DONNÉES BRUTES...")

    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                all_raw_data.extend(data)
                print(f"   ✅ {json_file}: {len(data)} lignes")
        except Exception as e:
            print(f"   ❌ {json_file}: {e}")

    if not all_raw_data:
        print("❌ Aucune donnée à exporter.")
        return None, None

    print(f"\n📊 TOTAL LIGNES BRUTES: {len(all_raw_data):,}")

    # Créer DataFrame SANS MODIFICATION
    df_raw = pd.DataFrame(all_raw_data)

    # Statistiques de base
    print(f"\n🔍 STATISTIQUES DES DONNÉES:")
    print(f"   • Lignes: {df_raw.shape[0]:,}")
    print(f"   • Colonnes: {df_raw.shape[1]}")
    print(f"   • Colonnes disponibles: {list(df_raw.columns)}")

    # Ajouter des métadonnées
    df_raw['_source_file'] = ""
    df_raw['_export_date'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    df_raw['_data_type'] = 'raw'
    df_raw['_row_id'] = range(1, len(df_raw) + 1)

    # Nom du fichier Excel
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    excel_filename = f'pharma_shop_raw_data_{timestamp}.xlsx'

    print(f"\n💾 SAUVEGARDE EN EXCEL...")

    # Créer un writer Excel
    with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
        # 1. Feuille principale avec toutes les données
        df_raw.to_excel(writer, sheet_name='DATA_BRUTE', index=False)
        print(f"   ✅ Feuille 'DATA_BRUTE': {len(df_raw)} lignes")

        # 2. Feuille de métadonnées
        metadata = {
            'Colonne': df_raw.columns.tolist(),
            'Type': [str(df_raw[col].dtype) for col in df_raw.columns],
            'Non Null': [df_raw[col].notna().sum() for col in df_raw.columns],
            'Null': [df_raw[col].isna().sum() for col in df_raw.columns],
            'Valeurs Uniques': [df_raw[col].nunique() for col in df_raw.columns]
        }
        df_metadata = pd.DataFrame(metadata)
        df_metadata.to_excel(writer, sheet_name='METADATA', index=False)
        print(f"   ✅ Feuille 'METADATA': {len(df_metadata)} colonnes")

        # 3. Feuille de statistiques par catégorie (si la colonne existe)
        if 'category' in df_raw.columns:
            category_stats = df_raw['category'].value_counts().reset_index()
            category_stats.columns = ['Category', 'Count']
            category_stats['Percentage'] = (category_stats['Count'] / len(df_raw) * 100).round(2)
            category_stats.to_excel(writer, sheet_name='STATS_CATEGORY', index=False)
            print(f"   ✅ Feuille 'STATS_CATEGORY': {len(category_stats)} catégories")

        # 4. Feuille avec échantillons
        samples = df_raw.head(100)  # 100 premières lignes comme échantillon
        samples.to_excel(writer, sheet_name='SAMPLES', index=False)
        print(f"   ✅ Feuille 'SAMPLES': 100 lignes d'échantillon")

        # 5. Feuille de log
        log_data = {
            'Timestamp': [datetime.now().strftime("%Y-%m-%d %H:%M:%S")],
            'Fichiers sources': [', '.join(json_files[:5]) + ('...' if len(json_files) > 5 else '')],
            'Total lignes': [len(df_raw)],
            'Total colonnes': [len(df_raw.columns)],
            'Taille estimée (MB)': [f"{(len(df_raw) * len(df_raw.columns) * 50) / (1024*1024):.2f}"],
            'Encodage': ['UTF-8'],
            'Nettoyage appliqué': ['AUCUN - Données brutes']
        }
        df_log = pd.DataFrame(log_data)
        df_log.to_excel(writer, sheet_name='LOG', index=False)
        print(f"   ✅ Feuille 'LOG': Informations d'export")

    # Vérifier la taille du fichier
    file_size_mb = os.path.getsize(excel_filename) / (1024*1024)

    print(f"\n✅ EXCEL SAUVEGARDÉ: {excel_filename}")
    print(f"📏 Taille du fichier: {file_size_mb:.2f} MB")

    # Aperçu
    print(f"\n📋 APERÇU DU FICHIER EXCEL:")
    print(f"   1. DATA_BRUTE    - Toutes les données ({len(df_raw):,} lignes)")
    print(f"   2. METADATA      - Métadonnées des colonnes")
    if 'category' in df_raw.columns:
        print(f"   3. STATS_CATEGORY - Statistiques par catégorie")
    print(f"   4. SAMPLES       - 100 premiers échantillons")
    print(f"   5. LOG           - Informations d'export")

    return df_raw, excel_filename

def export_simple_excel():
    """
    Version simple - un seul fichier JSON vers Excel
    """
    print("📄 EXPORT SIMPLE VERS EXCEL")
    print("="*60)

    # Chercher le dernier fichier JSON
    json_files = [f for f in os.listdir('.') if f.startswith('pharma_shop_') and f.endswith('.json')]

    if not json_files:
        print("❌ Aucun fichier JSON trouvé.")
        return None, None

    latest_json = max(json_files, key=os.path.getctime)
    print(f"📂 Chargement: {latest_json}")

    with open(latest_json, 'r', encoding='utf-8') as f:
        data = json.load(f)

    print(f"📊 Données: {len(data):,} lignes")

    # Convertir en DataFrame
    df = pd.DataFrame(data)

    # Nom du fichier
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    excel_file = f'pharma_shop_data_{timestamp}.xlsx'

    # Sauvegarder en Excel
    with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='Produits', index=False)

        # Ajouter un résumé
        summary = pd.DataFrame({
            'Statistique': ['Lignes', 'Colonnes', 'Date export', 'Source'],
            'Valeur': [len(df), len(df.columns),
                      datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                      latest_json]
        })
        summary.to_excel(writer, sheet_name='Résumé', index=False)

    print(f"\n✅ EXCEL CRÉÉ: {excel_file}")
    print(f"📏 Dimensions: {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

    return df, excel_file

def create_excel_with_formats():
    """
    Crée un Excel avec formats avancés (couleurs, filtres, etc.)
    """
    print("🎨 EXCEL AVEC FORMATS AVANCÉS")
    print("="*60)

    # Charger les données
    json_files = [f for f in os.listdir('.') if f.startswith('pharma_shop_') and f.endswith('.json')]
    if not json_files:
        print("❌ Aucun fichier JSON.")
        return None, None

    # Prendre les 3 derniers fichiers
    recent_files = sorted(json_files, key=lambda x: os.path.getctime(x), reverse=True)[:3]

    all_data = []
    for file in recent_files:
        with open(file, 'r', encoding='utf-8') as f:
            data = json.load(f)
            all_data.extend(data)

    df = pd.DataFrame(all_data)

    # Nom du fichier
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    excel_file = f'pharma_shop_formatted_{timestamp}.xlsx'

    # Créer le fichier Excel avec openpyxl pour plus de contrôle
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter

    wb = Workbook()

    # Feuille 1: Données complètes
    ws1 = wb.active
    ws1.title = "DONNÉES_COMPLÈTES"

    # Écrire les en-têtes
    headers = list(df.columns)
    for col_idx, header in enumerate(headers, 1):
        cell = ws1.cell(row=1, column=col_idx, value=header)
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill(start_color="366092", end_color="366092", fill_type="solid")
        cell.alignment = Alignment(horizontal="center")

    # Écrire les données
    for row_idx, row in enumerate(df.itertuples(index=False), 2):
        for col_idx, value in enumerate(row, 1):
            ws1.cell(row=row_idx, column=col_idx, value=value)

    # Ajuster la largeur des colonnes
    for col_idx, column in enumerate(ws1.columns, 1):
        max_length = 0
        column_letter = get_column_letter(col_idx)

        for cell in column:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass

        adjusted_width = min(max_length + 2, 50)
        ws1.column_dimensions[column_letter].width = adjusted_width

    # Ajouter des filtres
    ws1.auto_filter.ref = ws1.dimensions

    # Feuille 2: Résumé par catégorie
    if 'category' in df.columns:
        ws2 = wb.create_sheet("RÉSUMÉ_CATÉGORIES")

        stats = df['category'].value_counts().reset_index()
        stats.columns = ['Catégorie', 'Nombre']

        # En-têtes
        ws2['A1'] = 'Catégorie'
        ws2['B1'] = 'Nombre de produits'
        ws2['C1'] = 'Pourcentage'

        for cell in ['A1', 'B1', 'C1']:
            ws2[cell].font = Font(bold=True)

        # Données
        total = len(df)
        for idx, (cat, count) in enumerate(stats.itertuples(index=False), 2):
            ws2[f'A{idx}'] = cat
            ws2[f'B{idx}'] = count
            ws2[f'C{idx}'] = f"{(count/total*100):.1f}%"

        # Ajuster les colonnes
        ws2.column_dimensions['A'].width = 30
        ws2.column_dimensions['B'].width = 20
        ws2.column_dimensions['C'].width = 15

    # Feuille 3: Métadonnées
    ws3 = wb.create_sheet("MÉTADONNÉES")

    metadata = [
        ['Propriété', 'Valeur'],
        ['Date d export', datetime.now().strftime("%Y-%m-%d %H:%M:%S")],
        ['Source', 'pharma-shop.tn'],
        ['Nombre total', len(df)],
        ['Fichiers sources', ', '.join(recent_files)],
        ['Dernière mise à jour', datetime.now().strftime("%Y-%m-%d")],
        ['Note', 'Données brutes - Aucun nettoyage appliqué']
    ]

    for row_idx, row in enumerate(metadata, 1):
        for col_idx, value in enumerate(row, 1):
            cell = ws3.cell(row=row_idx, column=col_idx, value=value)
            if row_idx == 1:
                cell.font = Font(bold=True)

    # Sauvegarder
    wb.save(excel_file)

    print(f"\n✅ EXCEL AVEC FORMATS CRÉÉ: {excel_file}")
    print(f"📊 Données: {len(df):,} produits")

    return df, excel_file

# Menu principal
if __name__ == "__main__":
    print("🌐 PHARMA-SHOP.TN - EXPORT EXCEL")
    print("="*70)
    print("📊 Export des données brutes vers Microsoft Excel (.xlsx)")
    print("="*70)

    print("\n🎯 CHOISISSEZ UN FORMAT D'EXPORT:")
    print("   1. Excel complet (toutes données + métadonnées)")
    print("   2. Excel simple (dernier fichier seulement)")
    print("   3. Excel avec formats avancés (couleurs, filtres)")
    print("   4. Quitter")

    choice = input("\nVotre choix (1-4): ").strip()

    if choice == "1":
        print("\n" + "="*70)
        print("🔄 EXPORT COMPLET VERS EXCEL")
        print("="*70)
        df, filename = export_to_excel_raw()

        if df is not None:
            print(f"\n✅ EXPORT RÉUSSI!")
            print(f"📁 Fichier: {filename}")
            print(f"📊 {len(df):,} lignes exportées")

            # Option pour ouvrir
            open_it = input("\n📂 Ouvrir le fichier Excel? (o/n): ").lower()
            if open_it == 'o':
                try:
                    os.startfile(filename)  # Windows
                except:
                    try:
                        os.system(f'open "{filename}"')  # Mac
                    except:
                        print(f"📁 Fichier disponible: {os.path.abspath(filename)}")

    elif choice == "2":
        print("\n" + "="*70)
        df, filename = export_simple_excel()

        if df is not None:
            print(f"\n✅ FICHIER CRÉÉ: {filename}")

    elif choice == "3":
        print("\n" + "="*70)
        df, filename = create_excel_with_formats()

        if df is not None:
            print(f"\n✅ EXCEL FORMATTÉ CRÉÉ!")
            print(f"📁 {filename}")

    else:
        print("\n👋 Au revoir!")

🌐 PHARMA-SHOP.TN - EXPORT EXCEL
📊 Export des données brutes vers Microsoft Excel (.xlsx)

🎯 CHOISISSEZ UN FORMAT D'EXPORT:
   1. Excel complet (toutes données + métadonnées)
   2. Excel simple (dernier fichier seulement)
   3. Excel avec formats avancés (couleurs, filtres)
   4. Quitter

Votre choix (1-4): 3

🎨 EXCEL AVEC FORMATS AVANCÉS

✅ EXCEL AVEC FORMATS CRÉÉ: pharma_shop_formatted_20260201_213244.xlsx
📊 Données: 8,199 produits

✅ EXCEL FORMATTÉ CRÉÉ!
📁 pharma_shop_formatted_20260201_213244.xlsx
